### Using Recife dengue data only

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path 

In [2]:
current_dir = Path(Path.cwd()).parent.parent 
data_dir = current_dir / "data"
dataset_dir = data_dir / "processed/datasets"

In [3]:

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import h2o

def metricas_regressao(y_true, y_pred):
    return {
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred)
    }

def extrair_historico_tpot(tpot, fold):
    """
    Extrai o histÃ³rico de pipelines avaliados no TPOT.
    CompatÃ­vel com versÃµes que usam `evaluated_individuals`
    ou `evaluated_individuals_`.
    """
    if hasattr(tpot, "evaluated_individuals"):
        hist = tpot.evaluated_individuals

        if isinstance(hist, pd.DataFrame):
            df_hist = hist.copy()
        else:
            df_hist = pd.DataFrame.from_dict(hist, orient="index")

        df_hist["pipeline_id"] = df_hist.index
        df_hist["fold"] = fold
        return df_hist.reset_index(drop=True)

    elif hasattr(tpot, "evaluated_individuals_"):
        hist = tpot.evaluated_individuals_
        df_hist = pd.DataFrame.from_dict(hist, orient="index")
        df_hist["pipeline_id"] = df_hist.index
        df_hist["fold"] = fold
        return df_hist.reset_index(drop=True)

    return pd.DataFrame()


def extrair_leaderboard_h2o(aml, test_h2o, fold):
    """
    Extrai o leaderboard do H2O AutoML e calcula mÃ©tricas
    de teste externo para cada modelo listado.
    """
    lb = aml.leaderboard.as_data_frame().copy()
    lb["fold"] = fold

    metricas_modelos = []

    for model_id in lb["model_id"]:
        model = h2o.get_model(model_id)
        perf = model.model_performance(test_h2o)

        metricas_modelos.append({
            "model_id": model_id,
            "fold": fold,
            "rmse_test": perf.rmse(),
            "mae_test": perf.mae(),
            "r2_test": perf.r2()
        })

    df_metricas = pd.DataFrame(metricas_modelos)
    lb = lb.merge(df_metricas, on=["model_id", "fold"], how="left")
    return lb


In [4]:
from sklearn.model_selection import TimeSeriesSplit
ds = pd.read_csv(dataset_dir / "recife_dataset.csv")

ds["data"] = pd.to_datetime(ds["data"], errors="coerce")
ds["data_num"] = (ds["data"] - pd.Timestamp("1970-01-01")).dt.days
X = ds.drop(columns=['casos_dengue'])
y = ds['casos_dengue']
tscv = TimeSeriesSplit(n_splits=5)

## Auto-sklearn
    acadÃªmico e otimizado
> nao tem suporte para windows, rodaremos num conteiner
```cmd
docker build -t dengue -f Dockerfile.notebook .
docker run -it -v %cd%:/workspace dengue
```

In [5]:
"""import autosklearn.regression

automl = autosklearn.regression.AutoSklearnRegressor(
    time_left_for_this_task=120
)
automl.fit(X_train, y_train)
automl.score(X_test, y_test)"""

'import autosklearn.regression\n\nautoml = autosklearn.regression.AutoSklearnRegressor(\n    time_left_for_this_task=120\n)\nautoml.fit(X_train, y_train)\nautoml.score(X_test, y_test)'

## TPOT
    evolucao genÃ©tica

In [ ]:
from tpot import TPOTRegressor


# só numérico
X = X.select_dtypes(include=[np.number]).copy()

# limpa inf / nan
X = X.replace([np.inf, -np.inf], np.nan)
y = y.replace([np.inf, -np.inf], np.nan)

# imputação simples
X = X.fillna(X.median())
y = y.fillna(y.median())

resumo_tpot = []
historico_tpot = []

# TPOT 1.1.0 usa Dask internamente; em notebook/Windows,
# manter um worker em modo thread evita travas por multiprocessing.
tpot_params = {
    "search_space": "linear-light",
    "generations": 2,
    "population_size": 5,
    "cv": TimeSeriesSplit(n_splits=3),
    "verbose": 3,
    "random_state": 42,
    "scorers": ["neg_mean_squared_error"],
    "scorers_weights": [1],
    "preprocessing": True,
    "n_jobs": 1,
    "processes": False,
}

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    tpot = TPOTRegressor(**tpot_params)
    tpot.fit(X_train, y_train)

    y_pred = tpot.predict(X_test)
    resumo_fold = metricas_regressao(y_test, y_pred)
    resumo_fold["fold"] = fold
    resumo_fold["best_pipeline"] = str(getattr(tpot, "fitted_pipeline_", None))
    resumo_tpot.append(resumo_fold)

    hist_fold = extrair_historico_tpot(tpot, fold)
    if not hist_fold.empty:
        historico_tpot.append(hist_fold)

df_resumo_tpot = pd.DataFrame(
    resumo_tpot,
    columns=["fold", "rmse", "mae", "r2", "best_pipeline"]
)
df_historico_tpot = (
    pd.concat(historico_tpot, ignore_index=True)
    if historico_tpot else pd.DataFrame()
)

print("Melhor pipeline do último fold:")
print(getattr(tpot, "fitted_pipeline_", None))
print(f"Histórico TPOT coletado para {len(df_resumo_tpot)} folds.")


d:\pedro\Programacao\Projetos\dengue_prediction\env\lib\site-packages\stopit\__init__.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
d:\pedro\Programacao\Projetos\dengue_prediction\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\pedro\Programacao\Projetos\dengue_prediction\env\lib\site-packages\tpot\tpot_estimator\estimator.py:458: UserWarning: Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.
  warnings.warn("Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.")
d:\pedro\Programacao\Projetos\den

Generation:  1
Best mean_squared_error score: -370.89619883040933


Generation: 100%|██████████| 2/2 [00:16<00:00,  8.01s/it]

Generation:  2
Best mean_squared_error score: -37.44005847953216



d:\pedro\Programacao\Projetos\dengue_prediction\env\lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [14] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
d:\pedro\Programacao\Projetos\dengue_prediction\env\lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
d:\pedro\Programacao\Projetos\dengue_prediction\env\lib\site-packages\tpot\tpot_estimator\estimator.py:458: UserWarning: Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.
  warnings.warn("Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.")
d:\pedro\Programacao\Projetos\dengue_prediction\env\lib\site-packages\distributed\node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 53948 ins

...


d:\pedro\Programacao\Projetos\dengue_prediction\env\lib\site-packages\sklearn\feature_selection\_base.py:122: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(
d:\pedro\Programacao\Projetos\dengue_prediction\env\lib\site-packages\tpot\tpot_estimator\estimator.py:458: UserWarning: Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.
  warnings.warn("Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.")
d:\pedro\Programacao\Projetos\dengue_prediction\env\lib\site-packages\distributed\node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 53968 instead
  warnings.warn(
Generation:  50%|█████     | 1/2 [00:07<00:07,  7.19s/it]

Generation:  1
Best mean_squared_error score: -4472.487969483568


Generation: 100%|██████████| 2/2 [00:26<00:00, 13.21s/it]

Generation:  2
Best mean_squared_error score: -3854.9201874176924


Melhor pipeline do último fold:
Pipeline(steps=[('pipeline-1',
                 Pipeline(steps=[('impute_numeric', ColumnSimpleImputer())])),
                ('pipeline-2',
                 Pipeline(steps=[('standardscaler', StandardScaler()),
                                 ('selectfwe',
                                  SelectFwe(alpha=0.0025247539905)),
                                 ('featureunion-1',
                                  FeatureUnion(transformer_list=[('featureunion',
                                                                  FeatureUnion(transformer_list=[('quantiletransformer',
                                                                                                  QuantileTransformer(n_quanti...
                                                                                                  PowerTransformer())])),
                                                                 ('passthrough',
                                                    

## H2O AutoML 
    produÃ§Ã£o

In [7]:
from h2o.automl import H2OAutoML

h2o.init()

resumo_h2o = []
historico_h2o = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    train_df = X_train.copy()
    train_df["casos_dengue"] = y_train.values

    test_df = X_test.copy()
    test_df["casos_dengue"] = y_test.values

    train_h2o = h2o.H2OFrame(train_df)
    test_h2o = h2o.H2OFrame(test_df)

    x_cols = [c for c in train_h2o.columns if c != "casos_dengue"]
    y_col = "casos_dengue"

    aml = H2OAutoML(
        max_models=10,
        seed=42,
        nfolds=0
    )

    aml.train(
        x=x_cols,
        y=y_col,
        training_frame=train_h2o,
        leaderboard_frame=test_h2o
    )

    # melhor modelo do fold
    leader = aml.leader
    perf_leader = leader.model_performance(test_h2o)

    resumo_h2o.append({
        "fold": fold,
        "best_model_id": leader.model_id,
        "rmse": perf_leader.rmse(),
        "mae": perf_leader.mae(),
        "r2": perf_leader.r2()
    })

    # leaderboard completo do fold + mÃ©tricas no teste externo
    lb_fold = extrair_leaderboard_h2o(aml, test_h2o, fold)
    historico_h2o.append(lb_fold)

df_resumo_h2o = pd.DataFrame(resumo_h2o)
df_historico_h2o = (
    pd.concat(historico_h2o, ignore_index=True)
    if historico_h2o else pd.DataFrame()
)


Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; Java HotSpot(TM) 64-Bit Server VM (build 17.0.13+10-LTS-268, mixed mode, sharing)
  Starting server from D:\pedro\Programacao\Projetos\dengue_prediction\env\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\pedro\AppData\Local\Temp\tmpxyv1y6wy
  JVM stdout: C:\Users\pedro\AppData\Local\Temp\tmpxyv1y6wy\h2o_pedro_started_from_python.out
  JVM stderr: C:\Users\pedro\AppData\Local\Temp\tmpxyv1y6wy\h2o_pedro_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,05 secs
H2O_cluster_timezone:,America/Fortaleza
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,1 month and 8 days
H2O_cluster_name:,H2O_from_python_pedro_1xpb9r
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.977 Gb
H2O_cluster_total_cores:,4
H2O_cluster_allowed_cores:,4
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
10:14:37.539: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%


d:\pedro\Programacao\Projetos\dengue_prediction\env\lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
10:14:50.887: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%


...

d:\pedro\Programacao\Projetos\dengue_prediction\env\lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
10:15:07.484: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%


d:\pedro\Programacao\Projetos\dengue_prediction\env\lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
10:15:15.614: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%


d:\pedro\Programacao\Projetos\dengue_prediction\env\lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [8]:
print("=== RESUMO TPOT ===")
print(df_resumo_tpot)

print("\n=== HISTÃ“RICO TPOT ===")
print(df_historico_tpot.head())

print("\n=== RESUMO H2O ===")
print(df_resumo_h2o)

print("\n=== HISTÃ“RICO H2O ===")
print(df_historico_h2o.head())

=== RESUMO TPOT ===
   fold        rmse         mae         r2  \
0     1  115.528197   79.209251  -0.801224   
1     2  182.283432  169.955586 -11.578361   
2     3   62.470748   47.517421 -98.397421   
3     4   34.390584   26.702318  -2.537579   
4     5   44.823928   30.519934  -0.522634   

                                       best_pipeline  
0  Pipeline(steps=[('pipeline-1',\n              ...  
1  Pipeline(steps=[('pipeline-1',\n              ...  
2  Pipeline(steps=[('pipeline-1',\n              ...  
3  Pipeline(steps=[('pipeline-1',\n              ...  
4  Pipeline(steps=[('pipeline-1',\n              ...  

=== HISTÃ“RICO TPOT ===
   mean_squared_error Parents Variation_Function  \
0                 NaN     NaN                NaN   
1                 NaN     NaN                NaN   
2                 NaN     NaN                NaN   
3         -370.896199     NaN                NaN   
4                 NaN     NaN                NaN   

                                   

In [9]:
media_tpot = df_resumo_tpot[["rmse", "mae", "r2"]].mean().to_frame().T
media_tpot["biblioteca"] = "TPOT"

media_h2o = df_resumo_h2o[["rmse", "mae", "r2"]].mean().to_frame().T
media_h2o["biblioteca"] = "H2O AutoML"

comparacao_final = pd.concat([media_tpot, media_h2o], ignore_index=True)

print("\n=== COMPARAÃ‡ÃƒO FINAL ===")
print(comparacao_final)


=== COMPARAÃ‡ÃƒO FINAL ===
        rmse        mae         r2  biblioteca
0  87.899378  70.780902 -22.767444        TPOT
1  55.301449  38.754019  -1.100112  H2O AutoML


In [10]:
save_data_dir = data_dir / "result"
save_data_dir.mkdir(exist_ok=True)
df_resumo_tpot.to_csv(save_data_dir / "resumo_tpot.csv", index=False)
df_historico_tpot.to_csv(save_data_dir / "historico_tpot.csv", index=False)

df_resumo_h2o.to_csv(save_data_dir / "resumo_h2o.csv", index=False)
df_historico_h2o.to_csv(save_data_dir / "historico_h2o.csv", index=False)

comparacao_final.to_csv(save_data_dir / "comparacao_final_automl.csv", index=False)